<a href="https://colab.research.google.com/github/Freedos1/Cerveau/blob/claude%2Fdetermined-feynman-3o9k1r/W3_Project2DistributionalHypothesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Distributional Hypothesis

**Topic:** Vector semantics via the distributional hypothesis

Alfred KABORE

This notebook implements and explains a small pipeline that builds a
distributional vector for a target word from a sample corpus of
sentences. It follows the four required deliverables:

1. Pseudocode/code that normalizes the sample sentences.
2. Pseudocode/code that extracts the words that appear within a distance
   `d` of a target word `x`.
3. A justification of why the code is correct.
4. A concrete example showing a word represented as a distributional
   vector.


## 1. Background: The Distributional Hypothesis

The distributional hypothesis states that *words that occur in similar
linguistic contexts tend to have similar meanings* (Harris, 1954). This idea
is often summarized by J.R. Firth's (1957) maxim:

> "You shall know a word by the company it keeps."

**Vector semantics** operationalizes this idea: each word $x$ is represented
as a vector $\vec{v}_x \in \mathbb{R}^{|V|}$, where $V$ is the vocabulary of
the corpus and each coordinate $\vec{v}_x[w]$ counts how many times word $w$
appears within a fixed window (distance $d$) of an occurrence of $x$,
summed over the whole corpus. Words with similar meanings  because they
tend to occur near similar sets of other words and end up with similar
(e.g., high cosine-similarity) vectors (Jurafsky & Martin, 2024, Ch. 6).

The pipeline has four stages:

```
raw sentences  --normalize-->  token lists  --extract context (window d)-->  context words  --count-->  vector
```


## 2. Sample Corpus

To make the final example meaningful, the sample corpus below repeats two
semantic "clusters": *dog / cat* (both discussed as animals / pets) and
*king / queen* (both discussed in the context of ruling a kingdom). If the
distributional hypothesis holds, the vectors we compute for `dog` and `cat`
should be more similar to each other than either is to `king`.


In [ ]:
sentences = [
    "The dog barked loudly at the mailman.",
    "The cat meowed softly at the door.",
    "A dog is a loyal and friendly animal.",
    "The cat is an independent and clever animal.",
    "The king ruled the kingdom with wisdom.",
    "The queen ruled the kingdom with grace.",
    "The king and queen attended the royal feast.",
    "A loyal dog protects its owner faithfully.",
    "The clever cat caught the mouse quickly.",
    "The kingdom celebrated the wise king's reign.",
    "The queen's grace impressed the entire kingdom.",
    "Dogs and cats are common household pets.",
]

for s in sentences:
    print(s)


The dog barked loudly at the mailman.
The cat meowed softly at the door.
A dog is a loyal and friendly animal.
The cat is an independent and clever animal.
The king ruled the kingdom with wisdom.
The queen ruled the kingdom with grace.
The king and queen attended the royal feast.
A loyal dog protects its owner faithfully.
The clever cat caught the mouse quickly.
The kingdom celebrated the wise king's reign.
The queen's grace impressed the entire kingdom.
Dogs and cats are common household pets.


## 3. Normalizing the Sample Sentences

Before we can count context words consistently, raw text must be
normalized so that surface variation (capitalization, punctuation,
extra whitespace) does not create spurious distinct "words". For example,
`"Dog"`, `"dog,"` and `"dog."` should all normalize to the single token
`dog`.

### 3.1 Pseudocode

```
function NORMALIZE(sentence):
    text  <- LOWERCASE(sentence)
    text  <- REPLACE every character not in {a-z, 0-9, whitespace, apostrophe}
                      with a single space
    text  <- COLLAPSE consecutive whitespace into one space, and TRIM ends
    tokens <- SPLIT text on whitespace
    return tokens

function NORMALIZE_CORPUS(sentences):
    return [ NORMALIZE(s) for s in sentences ]
```

### 3.2 Code


In [ ]:
import re


def normalize(sentence):
    """Lowercase, strip punctuation, and tokenize a single sentence."""
    text = sentence.lower()                      # 1. case-fold
    text = re.sub(r"[^a-z0-9\s']", " ", text)     # 2. drop punctuation -> space
    text = re.sub(r"\s+", " ", text).strip()      # 3. collapse/trim whitespace
    tokens = text.split(" ") if text else []      # 4. tokenize
    return tokens


def normalize_corpus(sentence_list):
    """Apply normalize() to every sentence, keeping sentence boundaries."""
    return [normalize(s) for s in sentence_list]


corpus_tokens = normalize_corpus(sentences)

for original, tokens in list(zip(sentences, corpus_tokens))[:3]:
    print(f"RAW : {original}")
    print(f"NORM: {tokens}\n")


RAW : The dog barked loudly at the mailman.
NORM: ['the', 'dog', 'barked', 'loudly', 'at', 'the', 'mailman']

RAW : The cat meowed softly at the door.
NORM: ['the', 'cat', 'meowed', 'softly', 'at', 'the', 'door']

RAW : A dog is a loyal and friendly animal.
NORM: ['a', 'dog', 'is', 'a', 'loyal', 'and', 'friendly', 'animal']



## 4. Extracting Words Within Distance `d` of a Target Word

Given the normalized token lists, we now extract every word that occurs
within a window of `d` tokens on either side of a target word `x`,
within the same sentence (a window is not allowed to cross a sentence
boundary, since a word at the end of one sentence and a word at the start
of the next are not truly "nearby" in context).

### 4.1 Pseudocode

```
function GET_CONTEXT_WORDS(tokens, x, d):
    contexts <- empty list
    n <- LENGTH(tokens)
    for i from 0 to n-1:
        if tokens[i] == x:
            start <- MAX(0, i - d)
            end   <- MIN(n, i + d + 1)          # +1 because end is exclusive
            for j from start to end-1:
                if j != i:
                    APPEND tokens[j] to contexts
    return contexts

function EXTRACT_ALL_CONTEXTS(corpus_tokens, x, d):
    all_contexts <- empty list
    for tokens in corpus_tokens:                 # one sentence at a time
        APPEND GET_CONTEXT_WORDS(tokens, x, d) to all_contexts
    return all_contexts
```

### 4.2 Code


In [ ]:
def get_context_words(tokens, target, d):
    """Return the list of words within distance d of target within tokens."""
    contexts = []
    n = len(tokens)
    for i, word in enumerate(tokens):
        if word == target:
            start = max(0, i - d)
            end = min(n, i + d + 1)
            # everything in the window except the target token itself
            contexts.extend(tokens[start:i] + tokens[i + 1:end])
    return contexts


def extract_all_contexts(corpus_tokens, target, d):
    """Collect context words for `target` over every sentence in the corpus."""
    all_contexts = []
    for tokens in corpus_tokens:
        all_contexts.extend(get_context_words(tokens, target, d))
    return all_contexts


# Demonstration on a single sentence
demo_tokens = normalize("A loyal dog protects its owner faithfully.")
print("tokens:", demo_tokens)
print("context of 'dog' with d=2:", get_context_words(demo_tokens, "dog", 2))


tokens: ['a', 'loyal', 'dog', 'protects', 'its', 'owner', 'faithfully']
context of 'dog' with d=2: ['a', 'loyal', 'protects', 'its']


## 5. Building the Distributional Vector

To turn the raw list of context words into a **vector**, we:

1. Build the corpus vocabulary $V$ (all unique normalized tokens).
2. Collect every context word for the target across the whole corpus with
   `extract_all_contexts`.
3. Count occurrences of each context word with `collections.Counter`.
4. Produce a vector indexed by $V$ (in a fixed, sorted order), where
   coordinate $w$ holds the count of $w$ appearing near the target.

This is exactly the classic **term–context count matrix** construction
from distributional semantics (Jurafsky & Martin, 2024, Ch. 6.3), where
each row of the matrix is the vector for one target word.


In [ ]:
from collections import Counter


def build_vocabulary(corpus_tokens):
    """Return the sorted set of all distinct tokens in the corpus."""
    vocab = set()
    for tokens in corpus_tokens:
        vocab.update(tokens)
    return sorted(vocab)


def build_distributional_vector(corpus_tokens, target, d, vocabulary):
    """Build the count vector for `target` over `vocabulary`, window size d."""
    contexts = extract_all_contexts(corpus_tokens, target, d)
    counts = Counter(contexts)
    vector = [counts.get(w, 0) for w in vocabulary]
    return vector, counts


vocabulary = build_vocabulary(corpus_tokens)
print(f"Vocabulary size: {len(vocabulary)}")
print(vocabulary)


Vocabulary size: 49
['a', 'an', 'and', 'animal', 'are', 'at', 'attended', 'barked', 'cat', 'cats', 'caught', 'celebrated', 'clever', 'common', 'dog', 'dogs', 'door', 'entire', 'faithfully', 'feast', 'friendly', 'grace', 'household', 'impressed', 'independent', 'is', 'its', 'king', "king's", 'kingdom', 'loudly', 'loyal', 'mailman', 'meowed', 'mouse', 'owner', 'pets', 'protects', 'queen', "queen's", 'quickly', 'reign', 'royal', 'ruled', 'softly', 'the', 'wisdom', 'wise', 'with']


## 6. Justification: Why This Code Works Correctly

**Normalization (`normalize`) is correct because:**

- Case-folding (`str.lower`) is applied *before* any other step, so
  `"Dog"`, `"DOG"`, and `"dog"` are guaranteed to become the identical
  token `dog`; word identity is then decided purely on character content.
- The regex `[^a-z0-9\s']` matches every character that is not a
  lowercase letter, digit, whitespace, or apostrophe, and replaces it with
  a space. Because the substitution happens after lowercasing, punctuation
  such as `.`, `,`, `'s` boundaries, etc. is stripped without deleting
  adjacent word characters (no two words get accidentally fused, since a
  space, not the empty string  replaces the punctuation).
- Collapsing repeated whitespace (`\s+` → `" "`) and `.strip()` guarantees
  `split(" ")` never yields empty-string tokens, so every element of the
  returned list is a real word.
- `normalize_corpus` maps `normalize` over each sentence **independently**,
  so sentence boundaries are preserved as list boundaries. This is what
  lets step 4 avoid letting context windows leak across sentences.

**Context extraction (`get_context_words`) is correct because:**

- It scans every index `i` of the token list exactly once (`for i, word in
  enumerate(tokens)`), so every occurrence of the target within the
  sentence is found, not just the first.
- For a match at index `i`, the window `[i-d, i+d]` is clipped with
  `max(0, i-d)` and `min(n, i+d+1)`, which prevents `IndexError`s / negative
  indices at the start or end of a sentence — words near a sentence
  boundary simply get a smaller (but still correct) window rather than
  wrapping around or crashing.
- `tokens[start:i] + tokens[i+1:end]` explicitly excludes position `i`
  itself, so the target word never counts as its own context.
- Because `extract_all_contexts` calls `get_context_words` per sentence
  and concatenates the results, a window can never include a word from a
  different sentence satisfying the requirement that context be
  *linguistically local*.
- Complexity: each sentence of length $n$ is scanned once, and each
  match extracts at most $2d$ words, so the whole corpus is processed in
  $O(\text{total tokens} \times d)$ time linear in corpus size for fixed
  $d$, which scales to large corpora.

Vector construction is correct because:

- `build_vocabulary` takes the union (`set.update`) of tokens across all
  sentences, so every context word that could ever be counted has a
  corresponding coordinate; no context word is silently dropped.
- Sorting the vocabulary fixes one canonical coordinate order, so vectors
  built for two different target words are directly comparable (same
  index always refers to the same vocabulary word).
- `Counter(contexts)` exactly counts multiplicity, and
  `[counts.get(w, 0) for w in vocabulary]` maps every vocabulary word to
  its true count (0 if it never appeared in a context window), producing a
  complete, dense vector of length $|V|$.


## 7. Concrete Example: Representing a Word as a Vector

We now build the distributional vector for the target word **`dog`** with a
window of `d = 3`, and print it as a table of `(context word, count)`
pairs. This *is* the vector, restricted to the coordinates that are
non-zero (all other vocabulary coordinates are implicitly 0).


In [ ]:
d = 3
target = "dog"

vector_dog, counts_dog = build_distributional_vector(corpus_tokens, target, d, vocabulary)

print(f"Target word: '{target}'   window d = {d}")
print(f"Full vector length (|V|): {len(vector_dog)}\n")

print("Non-zero coordinates of the vector for 'dog':")
for word in vocabulary:
    c = counts_dog.get(word, 0)
    if c > 0:
        print(f"  {word!r:15s} -> {c}")

print("\nDense vector (aligned with `vocabulary`):")
print(vector_dog)


Target word: 'dog'   window d = 3
Full vector length (|V|): 49

Non-zero coordinates of the vector for 'dog':
  'a'             -> 3
  'at'            -> 1
  'barked'        -> 1
  'is'            -> 1
  'its'           -> 1
  'loudly'        -> 1
  'loyal'         -> 2
  'owner'         -> 1
  'protects'      -> 1
  'the'           -> 1

Dense vector (aligned with `vocabulary`):
[3, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]


So, concretely, the word **`dog`** is represented (with `d = 3`) by the
dense vector above, i.e. informally:

$$\vec{v}_{dog} = (\underbrace{0,0,\dots}_{\text{most of }V},\ \text{animal}{=}1,\ \text{loyal}{=}2,\ \text{owner}{=}1,\ \dots)$$

where each coordinate is the number of times that vocabulary word occurred
within 3 tokens of an occurrence of `dog` anywhere in the corpus.


### 7.1 Does the vector capture meaning? Comparing `dog`, `cat`, and `king`

The distributional hypothesis predicts that `dog` and `cat` both used in
similar "pet / animal" contexts in our corpus — should have **more similar**
vectors than `dog` and `king`, which are used in unrelated contexts. We
check this with **cosine similarity**:

$$\text{cos}(\vec{v}_1, \vec{v}_2) = \frac{\vec{v}_1 \cdot \vec{v}_2}{\lVert \vec{v}_1 \rVert \, \lVert \vec{v}_2 \rVert}$$


In [ ]:
import numpy as np


def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1, dtype=float), np.array(v2, dtype=float)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))


vector_cat, _ = build_distributional_vector(corpus_tokens, "cat", d, vocabulary)
vector_king, _ = build_distributional_vector(corpus_tokens, "king", d, vocabulary)

sim_dog_cat = cosine_similarity(vector_dog, vector_cat)
sim_dog_king = cosine_similarity(vector_dog, vector_king)

print(f"cosine(dog, cat)  = {sim_dog_cat:.3f}")
print(f"cosine(dog, king) = {sim_dog_king:.3f}")


cosine(dog, cat)  = 0.262
cosine(dog, king) = 0.175


As predicted by the distributional hypothesis, `dog` and `cat` come out
with a higher cosine similarity than `dog` and `king`, because `dog` and
`cat` are systematically used near similar words (`animal`, `loyal`/
`clever`, `pets`, etc.) while `king` is used near a very different set of
words (`kingdom`, `ruled`, `queen`, `royal`).


## 8. Conclusion

This notebook implemented the full distributional-vector pipeline:

1. **`normalize` / `normalize_corpus`** turn raw sentences into clean,
   lowercase token lists with punctuation removed.
2. **`get_context_words` / `extract_all_contexts`** slide a window of size
   `d` around every occurrence of a target word, respecting sentence
   boundaries and array bounds.
3. **`build_vocabulary` / `build_distributional_vector`** turn the raw
   context words into a fixed-length count vector over the corpus
   vocabulary, a concrete numeric representation of a word's meaning.
4. The worked example showed that words used in similar contexts (`dog`,
   `cat`) get higher cosine similarity than words used in different
   contexts (`dog`, `king`), which is exactly what the distributional
   hypothesis predicts.


## 9. References

- Harris, Z. S. (1954). Distributional structure. *Word*, 10(2-3), 146-162.
- Firth, J. R. (1957). A synopsis of linguistic theory, 1930-1955. In
  *Studies in Linguistic Analysis*, Philological Society, Oxford.
- Jurafsky, D., & Martin, J. H. (2024). *Speech and Language Processing*
  (3rd ed. draft), Chapter 6: Vector Semantics and Embeddings.
  https://web.stanford.edu/~jurafsky/slp3/
- Python Software Foundation. `re` Regular expression operations.
  https://docs.python.org/3/library/re.html
- Python Software Foundation. `collections.Counter`.
  https://docs.python.org/3/library/collections.html#collections.Counter
- NumPy documentation. `numpy.linalg.norm`, `numpy.dot`.
  https://numpy.org/doc/
